<a href="https://colab.research.google.com/github/nikita-boiko/PySpark_project/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
# Возможно, после входа в аккаунт вам потребуется заново выполнить код в этой ячейке.
kagglehub.login()
path = kagglehub.competition_download('riiid-test-answer-prediction')

In [ ]:
from pyspark.sql import SparkSession
import os
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark

df = spark.read.csv('/root/.cache/kagglehub/competitions/riiid-test-answer-\
prediction/train.csv', header=True, inferSchema=True)

df.printSchema()


In [ ]:
df.show()

In [ ]:
from pyspark.sql.types import IntegerType

df = df.withColumn('prior_question_had_explanation', df['prior_question_had_explanation'].cast(IntegerType()))
df.printSchema()

In [ ]:
spark.conf.set("spark.sql.ansi.enabled", "false")
df.pandas_api().isna().mean()

In [ ]:
df = df.dropna()
df.pandas_api().isna().sum()

In [ ]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
import pandas as pd

vector_col = 'corr_features'
assembler = VectorAssembler(inputCols=df.columns, outputCol=vector_col)
df_vector  = assembler.transform(df).select(vector_col)

matrix_row = Correlation.corr(df_vector, vector_col).collect()[0]

corr_matrix = matrix_row[0].toArray()

corr_matrix_df = pd.DataFrame(data=corr_matrix, columns=df.columns, index=df.columns)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme()

sns.heatmap(corr_matrix_df,
            xticklabels=corr_matrix_df.columns.values,
            yticklabels=corr_matrix_df.columns.values, cmap='Greens', annot=True)
plt.show()